# C02 · Premier League: team strengths, a title race and a fight with the bookmaker

| | |
|---|---|
| **Difficulty** | ★★★☆☆ |
| **Time** | 3-4 hours |
| **Data** | football-data.co.uk: every Premier League match of 2024/25 and 2025/26, with Bet365 odds |
| **Skills** | Poisson regression with crossed group effects · diagnosing non-identifiability · `pm.ZeroSumNormal` · hierarchical shrinkage · posterior predictive checks on quantities the model never saw · simulating a season from the posterior predictive · proper scoring rules against a market benchmark |

## The brief

You have joined the analytics desk of a sports media company. The editor wants two things:

1. **Team strength estimates** - attack and defence ratings with honest uncertainty, not a
   league table with extra steps.
2. **Probabilistic forecasts** - "Arsenal have an X% chance of the title, West Ham a Y%
   chance of going down" - that the desk can publish mid-season without embarrassing itself.

The head of the desk is a sceptic: *"Bookmakers do this for a living. Show me your numbers
next to theirs on matches your model has not seen, then we talk."*

You will build the model on the complete 2024/25 season, then deploy it on 2025/26 as it
stood on **13 February 2026** - 26 rounds played, 12 to go - and forecast the rest. We know
how the season ended, so we can mark your homework.

## How this notebook works

- Each task states **what to deliver**, not how. Write your code in the `YOUR CODE HERE` cells.
- Stuck? `h.hint("task2")` reveals hints one level at a time: *nudge → approach → code skeleton*.
  Try to get by on nudges.
- `h.check("task2", home=...)` compares your numbers with the reference solution.
- A full worked solution lives in `notebooks/solutions/`. Open it only when you are done (or truly stuck).

In [ ]:
import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymc as pm
from scipy import stats

from pymc_challenges import Hints, data

RANDOM_SEED = 2526
rng = np.random.default_rng(RANDOM_SEED)
az.style.use("arviz-variat")

h = Hints("C02")
h.tasks()

## The data

One row per match. The columns you need: `HomeTeam`, `AwayTeam`, full-time home and away
goals `FTHG` and `FTAG`, the full-time result `FTR` (`H`, `D` or `A`), and Bet365's
pre-match decimal odds `B365H`, `B365D`, `B365A` (a winning 1-unit bet at odds 2.5 pays
back 2.5). There are many more columns (shots, cards, other bookmakers); ignore them or
use them in "Going further".

In [ ]:
COLUMNS = ["Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR", "B365H", "B365D", "B365A"]


def prepare(df):
    df = df[COLUMNS].copy()
    df["Date"] = pd.to_datetime(df.Date, dayfirst=True)
    return df.sort_values("Date", kind="stable").reset_index(drop=True)


data.describe("epl_2425")
epl25 = prepare(data.load("epl_2425"))
epl25.head()

A helper for the bookkeeping, so you can spend your time on the modelling. Ties on points
are broken by goal difference, then goals scored.

In [ ]:
def league_table(df):
    """League table (points, goals for/against, goal difference) from a frame of matches."""
    home = pd.DataFrame({"team": df.HomeTeam, "gf": df.FTHG, "ga": df.FTAG})
    away = pd.DataFrame({"team": df.AwayTeam, "gf": df.FTAG, "ga": df.FTHG})
    long = pd.concat([home, away])
    long["played"] = 1
    long["pts"] = np.select([long.gf > long.ga, long.gf == long.ga], [3, 1], 0)
    table = long.groupby("team")[["played", "pts", "gf", "ga"]].sum()
    table["gd"] = table.gf - table.ga
    table = table.sort_values(["pts", "gd", "gf"], ascending=False)
    table.insert(0, "pos", range(1, len(table) + 1))
    return table


league_table(epl25)

## Task 0 · Look at the data

Use the 2024/25 season.

**Deliver**
1. How large is home advantage - in goals and in results?
2. Are goals plausibly Poisson? Compare the observed distribution of home goals and of
   away goals with a Poisson of the same mean, and compare means with variances.
3. One sentence: is a Poisson likelihood a defensible starting point?

In [ ]:
# YOUR CODE HERE

In [ ]:
# h.hint("task0")
# h.check("task0", home_goals_mean=..., away_goals_mean=...)   # goals per match, 2024/25

## Task 1 · The textbook model

The classic model for football scores gives every team an **attack** and a **defence**
rating. For a match between home team $h$ and away team $a$:

$$
\begin{aligned}
\text{home goals} &\sim \text{Poisson}(\lambda_h), & \log \lambda_h &= \mu + \eta + \text{att}_h - \text{def}_a \\
\text{away goals} &\sim \text{Poisson}(\lambda_a), & \log \lambda_a &= \mu + \text{att}_a - \text{def}_h
\end{aligned}
$$

with $\mu$ an intercept and $\eta$ the home advantage. A colleague suggests
"uninformative priors, let the data speak": independent `Normal(0, 3)` on $\mu$, on $\eta$
and on every $\text{att}$ and $\text{def}$.

**Deliver**
1. Fit exactly this model to 2024/25 with default sampler settings, using `coords`/`dims`
   for teams.
2. Read the diagnostics. Something is wrong. **Explain what, in terms of the model** - not
   in terms of the sampler - and produce one plot or table that demonstrates your explanation.
3. Which quantities does this posterior pin down well in spite of the problem, and which
   not at all? Give one example of each, with numbers.

In [ ]:
# YOUR CODE HERE

In [ ]:
# h.hint("task1")
# h.check("task1", sd_attack_difference=...)   # posterior sd of attack[Liverpool] - attack[Southampton]

## Task 2 · An identified, hierarchical model

**Deliver**
1. A reformulated model in which every parameter is identified **by construction** (not by
   luck of the prior), and in which the *spread* of attacking and of defensive ability
   across the league is learned from the data rather than fixed by you.
2. Priors you can defend, backed by a prior predictive check on a scale a football fan
   would recognise.
3. Clean diagnostics, and a comparison with Task 1: what happened to the effective sample size?
4. A plot of attack and defence ratings with uncertainty. Then compare your attack ratings
   with the naive estimate "log of goals scored relative to the league average". Which way
   does the model move teams, and why?

In [ ]:
# YOUR CODE HERE

In [ ]:
# h.hint("task2")
# h.check("task2", home=..., sigma_att=..., sigma_def=...)   # posterior means, 2024/25

## Task 3 · Check the model against things it was not fitted to

The model was fitted to home goals and away goals, separately. Football is about
**results**. A model can match both goal distributions and still get the things people
care about wrong.

**Deliver** posterior predictive checks, each with a verdict, for
1. the **share of draws** in a season,
2. the frequency of the most common **scorelines** (0-0, 1-1, 1-0, ...),
3. one further statistic of your choice that the likelihood does not target directly.

Does anything fail badly enough to matter for forecasting results?

In [ ]:
# YOUR CODE HERE

In [ ]:
# h.hint("task3")
# h.check("task3", draw_share_replicated=...)   # mean share of draws across replicated 2024/25 seasons

## Task 4 · Forecast the run-in

Now the live season. It is 13 February 2026: every team has played 26 of 38 matches.
`played` is what you may fit to; `remaining` holds the 120 fixtures still to come. (It
also holds their results and odds. Those are for marking - do not let the model see them.)

In [ ]:
data.describe("epl_2526")
epl26 = prepare(data.load("epl_2526"))

CUTOFF = pd.Timestamp("2026-02-13")
played = epl26[epl26.Date < CUTOFF].reset_index(drop=True)
remaining = epl26[epl26.Date >= CUTOFF].reset_index(drop=True)

print(f"{len(played)} matches played, {len(remaining)} remaining")
league_table(played)

**Deliver**
1. Your Task 2 model fitted to `played`, with clean diagnostics.
2. Simulations of the 120 remaining fixtures from the **posterior predictive** (parameter
   uncertainty *and* match-day randomness), turned into complete final league tables -
   one table per posterior draw, with the usual tie-breakers.
3. For every team: expected final points with a 94% interval, **P(title)**, **P(top 4)**,
   **P(relegation)** (bottom three).
4. The reckoning: compare with `league_table(epl26)`. Were the favourites right? How many
   of the 20 final points totals fall inside your 94% intervals, and how many should?

In [ ]:
# YOUR CODE HERE

In [ ]:
# h.hint("task4")
# h.check("task4", p_title_arsenal=..., p_relegation_west_ham=..., expected_points_arsenal=...)

## Task 5 · Your numbers next to the bookmaker's

The sceptic's test. For each of the 120 held-out matches, the model and the bookmaker each
give three probabilities (home win, draw, away win), and then one of the three happened.

Two proper scoring rules, both "lower is better", averaged over matches, where $p_{ik}$ is
the probability given to outcome $k$ in match $i$ and $o_{ik}$ is 1 if it happened, else 0:

$$\text{log score} = -\frac{1}{N}\sum_i \log p_{i,\text{observed}} \qquad\qquad
\text{Brier} = \frac{1}{N}\sum_i \sum_{k \in \{H,D,A\}} (p_{ik} - o_{ik})^2$$

**Deliver**
1. Model probabilities for H/D/A for every held-out match, from the Task 4 posterior.
2. Bookmaker probabilities from the Bet365 odds. Raw inverse odds do not sum to one; how
   big is the bookmaker's margin (the "overround"), and what do you do about it?
3. A reference forecaster that knows nothing about teams.
4. A table of both scores for all three forecasters - and a statement about whether 120
   matches are enough to separate them. Quantify it.

In [ ]:
# YOUR CODE HERE

In [ ]:
# h.hint("task5")
# h.check("task5", overround=..., log_score_bookmaker=..., log_score_model=...)

## Task 6 · Would you bet on it?

The head of desk reads your table and asks the natural follow-up: *"Fine, the market is
better on average. But when your model disagrees with it - is there money in that? And
what does the market know that your model does not?"*

**Deliver**
1. A back-test on the held-out matches: stake 1 unit on every outcome where your model
   sees positive expected value at the Bet365 odds (the real odds, margin included). Number of
   bets, profit, return on investment - **with an uncertainty interval** on the ROI.
2. A diagnosis of *how* the model's probabilities differ from the market's. Is the model
   more or less opinionated? If you mix the two forecasts, `w * model + (1 - w) * market`,
   which `w` scores best on the held-out matches, and what does that tell you?
3. A recommendation in three sentences. What is the model good for, what is it not good
   for, and what would you add first?

In [ ]:
# YOUR CODE HERE

In [ ]:
# h.hint("task6")
# h.check("task6", best_blend_weight=...)   # weight on the model in the best model/market mixture

## Going further

- **Last season as a prior.** Fit both seasons jointly, with a 2025/26 rating centred on
  a fraction of the 2024/25 rating and a separate prior for promoted teams. Does it score
  better on the held-out matches? (A quick attempt of ours scored 1.056 against 1.052 for
  the single-season model at round 26 - no better - and its sampler diagnostics needed work.
  Does it help at round 8, when the current season says little?)
- **Dependent scores.** Task 3 found home and away goals more negatively correlated than
  the model allows, and the run-in had more draws than either forecaster expected. Add the
  low-score correction of Dixon & Coles (1997) with `pm.Potential` or `pm.CustomDist`, or a
  match-level random effect shared by both sides with opposite signs. Does the log score improve?
- **Time-varying strength.** Chelsea and Burnley broke the model in Task 4. Let
  ratings follow a `pm.GaussianRandomWalk` over match rounds. Is there evidence that form
  is real, or is it noise?
- **Better data.** Shots on target (`HST`, `AST`) are about three times as plentiful as
  goals. Model them jointly with goals and see whether ratings sharpen faster early in the
  season.

In [ ]:
h.progress()